In [7]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import rasterio
import pyproj as proj

from flux_footprint.utils import find_transform, mask_fp_cutoff
from flux_footprint import calc_footprint_FFP as fp_1
from flux_footprint import calc_footprint_FFP_climatology as fp_2
print("Kljun FFP model imported successfully!")

dir_output = r"C:\GitHub\flux-footprint-py\outputs"
if not os.path.exists(dir_output):
    os.makedirs(dir_output)
    print(f"{dir_output} is created successfully.")
else:
    print(f"{dir_output} already exists.")

Kljun FFP model imported successfully!
C:\GitHub\flux-footprint-py\outputs already exists.


In [ ]:
file_path = r'C:\GitHub\flux-footprint-py\example_data\example_meteo_days.txt'
df = pd.read_csv(file_path, sep='\t') 

# Retrieve data from the DataFrame
zm = df['zm'].tolist()
hc = df['hc'].tolist()
z0 = df['z0'].tolist()        
umean = df['umean'].tolist()  
h = df['h'].tolist()          
ol = df['ol'].tolist()        
sigmav = df['sigmav'].tolist()
ustar = df['ustar'].tolist()
wind_dir = df['wind_dir'].tolist() 

# Set up parameters for footprint calculation
domain = None
dx, dy = 3, 3
nx, ny = 200, 200
rs = [0.9]      # 这里的 rs 在模型中通常建议传列表，如 [0.9]
crop = 0
fig = 1

# Site coordinates
latitude = 37.0334
longitude = -119.2622
cutoff = 0.9

filename = "US-xSP_footprint_test_delete_days.tif"

In [11]:
FFP_2 = fp_2.FFP_climatology(zm=zm,
                             z0=z0,
                             umean=umean,
                             h=h,
                             ol=ol,
                             sigmav=sigmav,
                             ustar=ustar,
                             wind_dir=wind_dir,
                             dx=dx,
                             dy=dy,
                             rs=rs,
                             crop=crop,
                             fig=0)

# Save results as GeoTIFF
# convert locations into UTM Zone 10N.
cor_x = round(longitude,4)
cor_y = round(latitude,4)
site_cord = (cor_x,cor_y) 
# WGS 84 Geographic Coordinate System
prj_gcs = proj.Proj(init='EPSG:4326')
# WGS 84 / UTM Zone 10N Projected Coordinate System
prj_pcs = proj.Proj(init='EPSG:32610')
(site_x,site_y) = proj.transform(prj_gcs,prj_pcs,*site_cord)

output_data = None
f_2d = np.array(FFP_2['fclim_2d'])
x_2d = np.array(FFP_2['x_2d']) + site_x
y_2d = np.array(FFP_2['y_2d']) + site_y
f_2d = f_2d*dx**2

#Calculate affine transform for given x_2d and y_2d
affine_transform = find_transform(x_2d,y_2d)
file_output_path = os.path.join(dir_output, filename)
new_dat = rasterio.open(file_output_path,'w',driver='GTiff',dtype=rasterio.float64,
                        count=1,height=f_2d.shape[0],width=f_2d.shape[1],
                        transform=affine_transform,
                        crs='+proj=utm +zone=10 +ellps=WGS84 +datum=WGS84 +units=m +no_defs',
                        nodata=0.00000000e+000)
# #Mask out points that are below a % threshold (defaults to 90%)
f_2d = mask_fp_cutoff(f_2d, cutoff=cutoff)
#Write the new band
new_dat.write(f_2d,1)
new_dat.close()
new_dat = None


Alert(0017):
 Only one value of zm passed. Using it for all footprints.
 Execution continues.

Alert(0013):
 Using z0, ignoring umean if passed.
 Execution continues.

Calculating footprint  1  of  1


c:\tools\Anaconda\envs\test_footprint_delete\Lib\site-packages\pyproj\crs\crs.py:143: FutureWarning: '+init=<authority>:<code>' syntax is deprecated. '<authority>:<code>' is the preferred initialization method. When making the change, be mindful of axis order changes: https://pyproj4.github.io/pyproj/stable/gotchas.html#axis-order-changes-in-proj-6
  in_crs_string = _prepare_from_proj_string(in_crs_string)
c:\tools\Anaconda\envs\test_footprint_delete\Lib\site-packages\pyproj\crs\crs.py:143: FutureWarning: '+init=<authority>:<code>' syntax is deprecated. '<authority>:<code>' is the preferred initialization method. When making the change, be mindful of axis order changes: https://pyproj4.github.io/pyproj/stable/gotchas.html#axis-order-changes-in-proj-6
  in_crs_string = _prepare_from_proj_string(in_crs_string)
C:\Users\Raymond G\AppData\Local\Temp\ipykernel_34260\391583584.py:24: FutureWarning: This function is deprecated. See: https://pyproj4.github.io/pyproj/stable/gotchas.html#upgradi